In [ ]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def removekey(d, *keys):
    r = dict(d)
    for _ in keys:
        del r[_]
    return r

In [ ]:
# make the marks using precipitation data(rules : sepearte hourly rainfall events by six consective zeros)    
def making_marks(precipitation):
    # first create an empty mark arary.
    mark = np.zeros_like(precipitation)
    # give values to the mark array.
    for i in range(len(precipitation)):
        if i < 6:
            mark[i] = 0
        else:
            if precipitation[i] > 0:
                if sum(precipitation[i-6:i]) > 0:
                    mark[i] = mark[i-1]
                else:
                    mark[i] = mark[i-1] + 1
            else:
                mark[i] = mark[i-1]
    return mark

In [ ]:
# according to the event mark, get to sum of x for each event, and rank the sum from highest to lowest. 
def ranking(df, x, num):
    rank = np.zeros(num)
    for i in range(num):
        rank[i] = sum(df[df.mark==i][x])
    return sorted(rank, reverse=True)

In [ ]:
from collections import OrderedDict
# from pastas import * # timeseries analysis tool
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

class Analyse(object):
    
    def __init__(self, filename, num_year=30, inflow_factor=1):
        self.name = filename
        # automatically create output name according to inputname: first remove ".csv" then add "_results.csv"
        self.output_name  = ''.join(list(self.name)[:-4])+"_results.csv"
        self.df = pd.read_csv(self.name,)
        self.df = self.df.fillna(0)  # fill the Na (the first row)
        self.dictionary = self.df.to_dict('list')
        self.num_year = num_year
        self.inflow_factor = inflow_factor
        
        # making event marks according to precipitation (6 consective zeros as separation),
        # create an extra colume named "Mark"
        self.df["mark"] = making_marks(self.df["P_atm"])
        self.measure_dictionary = removekey(self.dictionary, "Unnamed: 0", "Date", "P_atm", "Baseline")  # remove irrelevent columnes.
        
    def makingranks(self,):
        # unchanged,
        emp = dict()
        emp["Rank_P"] = ranking(self.df, "P_atm", int(max(self.df.mark)+1))
        emp["T_list"] = [(self.num_year + 1)/m for m in range(1, len(emp["Rank_P"])+1)]  # create T list. 31, ...
        # rank runoff on the baseline case
        emp["Rank_baseline"] = ranking(self.df, "Baseline", int(max(self.df.mark)+1))
        for key in self.measure_dictionary.keys():
            emp[key] = ranking(self.df, key, int(max(self.df.mark)+1))
        data = pd.DataFrame.from_dict(emp)
        return data
    # print(m.makingranks()) to see all these rankings.
    
    def save_to_csv(self,):
        self.makingranks().to_csv(self.output_name)
        
    def plotting(self, measure_name, addition_name, curve_to_plot, values_to_plot, xlim_down=5, xlim_up=20,plotfactorlabel=True,msg1="CP",msg2="20",msg3="OW",msg4="SWDS",msg5="Drainage capacity: 0.5 storage capacity"):
        self.data = self.makingranks()
        
        plt.figure(figsize=(16,9))  # figsize=(9,6)
        plt.semilogy(self.data.Rank_P, self.data.T_list, "b--", label="Rainfall", ms=2)
        plt.semilogy(self.data.Rank_baseline, self.data.T_list, "k-", label="Baseline runoff", ms=2)
        measures_rank_dictionary = removekey(self.data.to_dict('list'), "Rank_P", "Rank_baseline", "T_list")
        
        for key in measures_rank_dictionary.keys():
            plt.semilogy(measures_rank_dictionary[key], self.data.T_list, label="ED" + str(round(float(key)/self.inflow_factor))+ " runoff", ms=2)
        
        x=np.linspace(0,100,100)
        # plt.legend(loc='best',frameon=False)
        plt.annotate('Location '+ msg1, xy=(0.7,0.15), xycoords='axes fraction')
        plt.annotate('Inflow factor '+msg2, xy=(0.7,0.125), xycoords='axes fraction')
        if msg3 is not None:
            plt.annotate('Controlled runoff to '+msg3, xy=(0.7, 0.1), xycoords='axes fraction')
        plt.annotate('Uncontrolled runoff to '+msg4,xy=(0.7, 0.075), xycoords='axes fraction')
        if msg5 is not None:
            plt.annotate(msg5,xy=(0.7, 0.05), xycoords='axes fraction')
        plt.annotate('ED - (static) effective depth',xy=(0.7, 0.025), xycoords='axes fraction')
        
        if plotfactorlabel:
            for value in values_to_plot:
                base_series = self.data["Rank_baseline"]
                curve_series = self.data[curve_to_plot]  # make it optional
                base_id = base_series.size - np.searchsorted(base_series[::-1], value, side="right")
                curve_id = curve_series.size - np.searchsorted(curve_series[::-1], value, side="right")
        #         print(self.data["T_list"].values[ed_id][0])
        #         print(self.data["T_list"].iloc[ed_id].values[0]) 
                
#                 print("Case: ", value)
#                 print("Base:")
#                 print("T_above baseline index", base_id)
                value_small1 = self.data["T_list"].values[base_id-1][0] # 3.1
#                 print("T_above", value_small1)
                value_small2 = self.data["T_list"].values[base_id][0]  # 2.8
#                 print("T_below", value_small2)
                Real_T_small = value_small1 - (value_small1 - value_small2) * (self.data["Rank_baseline"].values[base_id-1][0] - value)/ (self.data["Rank_baseline"].values[base_id-1][0]- self.data["Rank_baseline"].values[base_id][0])
#                 print("T_real baseline", Real_T_small)
#                 print("Curve:")
                value_large1 = self.data["T_list"].values[curve_id-1][0]
#                 print("T_above", value_large1)
                value_large2 = self.data["T_list"].values[curve_id][0]
#                 print("T_below", value_large2)
                Real_T_large = value_large1 - (value_large1 - value_large2) * (self.data[curve_to_plot].values[curve_id-1][0] - value)/ (self.data[curve_to_plot].values[curve_id-1][0]- self.data[curve_to_plot].values[curve_id][0])
#                 print("T_real curve", Real_T_large)
                difference = Real_T_large - Real_T_small
                # add interpolation
                factor = round(Real_T_large / Real_T_small,2)
                # plot one arrow
                plt.arrow(x=self.data["Rank_baseline"].values[base_id][0],
                          y=self.data["T_list"].values[base_id][0],
                          dx=0,
                          dy= 0.8 * difference,
                          fc="k", ec="k",head_width=0.3, head_length=0.2 * difference, linestyle="--")
                texttoprint = f"T={round(Real_T_small,4)} -> T={round(Real_T_large,4)}, factor = {factor}"
                plt.text(self.data["Rank_baseline"].values[base_id][0]+0.3, self.data["T_list"].values[base_id][0] + 0.4*difference, 
                         texttoprint,bbox={'facecolor':'red', 'alpha':0.66,'boxstyle':'round'},size=8)
    # {'fontname':'Arial', 'size':'16', 'color':'black', 'weight':'normal','verticalalignment':'bottom'}
    #         ax.text(3, 8, 'boxed italics text in data coords', style='italic',bbox={'facecolor':'red', 'alpha':0.5, 'pad':10})       
        plt.legend(loc='lower right', frameon=False)
        plt.xlabel("Depth (mm)")
        plt.ylabel("Return period (year)")
        plt.title("Event-based rainfall depth and runoff depth for "+ measure_name +  " (1988-2017)")
        plt.xlim(xlim_down, xlim_up)

        # add grid
        ax = plt.gca()
        ax.yaxis.grid(linestyle='--', linewidth=0.5, which='both')
        ax.xaxis.grid(linestyle='--', linewidth=0.5, which='both')

        plt.savefig("figures/"+ addition_name + measure_name+".png")
        # remove to show figure
        plt.ioff()

Example 1: Rain barrel

In [ ]:
m = Analyse("data/ep_rainbarrel.csv", num_year = 30, inflow_factor = 20)
m.plotting(measure_name="Rain barrel (test) ", addition_name="figs/part_", curve_to_plot="200", values_to_plot=[5,10, 20] ,xlim_down=0, xlim_up=40)

Example 2: urban wetland

In [ ]:
m = Analyse("data/ep_urbanwetland.csv", num_year=30, inflow_factor=10)
m.plotting(measure_name="Urban wetland", addition_name="figs/s10_", 
           curve_to_plot="100", values_to_plot=[5,10,20,30], 
           xlim_down=0, xlim_up=40,
           plotfactorlabel=True,
           msg1="CP",msg2="10",msg3="OW",msg4="SWDS",msg5="Drainage resistance 2d")
m.plotting(measure_name="Urban wetland", addition_name="figs/s20_", 
           curve_to_plot="200", values_to_plot=[5,10,20,30], 
           xlim_down=0, xlim_up=40,
           plotfactorlabel=True,
           msg1="CP",msg2="10",msg3="OW",msg4="SWDS",msg5="Drainage resistance 2d")
m.plotting(measure_name="Urban wetland", addition_name="figs/l_", 
           curve_to_plot="100", values_to_plot=[5,10,20,30], 
           xlim_down=0, xlim_up=100,
           plotfactorlabel=False,
           msg1="CP",msg2="10",msg3="OW",msg4="SWDS",msg5="Drainage resistance 2d")